# Sistema de agentes del Deliverable 3 — Phi-3.5-mini

Ejecuta las celdas **en orden, de arriba abajo**.

| sección | qué hace | cuándo |
|---|---|---|
| 1 a 4 | GPU, clon, autopruebas, Drive, carga del modelo | siempre |
| 5 | sistema completo sobre el conjunto de desarrollo | cada vez que se ajusta un prompt |
| 6 | todas las configuraciones sobre los conjuntos de prueba | **una vez**, tras subir el protocolo |
| 7 | bajar los resultados en un zip | al terminar |


## 1 · GPU, dependencias y repositorio
Debe decir `cuda: True` y mostrar una **Tesla T4**. Si no: *Entorno de ejecución* →
*Cambiar tipo de entorno de ejecución* → **T4 GPU**.

`COMMIT` trae el commit que sube `PROTOCOLO_D3.md`, con el que se hizo la corrida final. Así
el notebook corre el mismo código y los mismos prompts que se midieron. `COMMIT = ''` clona la
última versión de `main`, que sirve solo para desarrollo: el runner se niega a mezclar filas
de otro commit en un archivo de resultados ya guardado.


In [ ]:
COMMIT = '9158550'

%cd /content
!nvidia-smi -L
!pip -q install transformers==5.17.0 accelerate==1.14.0 bitsandbytes==0.50.2
!rm -rf /content/proyecto && git clone -q https://github.com/Luquitas02/genai-inscripcion-ramos /content/proyecto
import subprocess
if COMMIT:
    subprocess.run(['git', '-C', '/content/proyecto', 'checkout', '-q', COMMIT], check=True)
!git -C /content/proyecto log -1 --format='commit %h  %ad  %s'
import torch; print('cuda:', torch.cuda.is_available())
from importlib.metadata import version
for paquete in ('torch', 'transformers', 'accelerate', 'bitsandbytes'):
    print(f'{paquete}=={version(paquete)}')

## 2 · Autopruebas
No usan la GPU. Tienen que terminar en `TODO EN ORDEN`; si alguna falla, no sigas.


In [ ]:
import subprocess, sys, os
os.chdir('/content/proyecto/scripts')
pruebas = (['verificador.py'], ['ficha.py'], ['pipeline.py'], ['runner_d2.py', '--pruebas'],
           ['herramientas.py'], ['runner_d3.py', '--pruebas'])
for args in pruebas:
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    ultima = (r.stdout.strip().splitlines() or ['(sin salida)'])[-1]
    print(f'[{"OK  " if r.returncode == 0 else "FALLA"}] {args[0]:16} {ultima[:60]}')
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit('*** NO SIGAS: falló ' + args[0] + ' ***')
print('TODO EN ORDEN')

## 3 · Google Drive
**No te saltes esta celda.** Cada corrida se copia a Drive apenas termina, en la carpeta
`genai_d3`. Sin Drive, una sesión caída pierde todo lo medido.


In [ ]:
from google.colab import drive
import shutil
from pathlib import Path
drive.mount('/content/drive')
RESPALDO = Path('/content/drive/MyDrive/genai_d3')
(RESPALDO / 'resultados').mkdir(parents=True, exist_ok=True)
(RESPALDO / 'trazas').mkdir(parents=True, exist_ok=True)

def respaldar(*rutas):
    for r in rutas:
        r = Path(r)
        shutil.copy(r, RESPALDO / r.parent.name / r.name)
    print('copiado a Drive:', ', '.join(Path(r).name for r in rutas))

print('Drive listo:', RESPALDO)

## 4 · Carga del modelo
Phi-3.5-mini en 4 bits, 256 tokens de salida, generación greedy: lo mismo que el D2, con la
generación cortada al cerrar el primer JSON (no cambia la respuesta que lee el agente, solo
evita generar la explicación que se descartaba). Tarda entre tres y cinco minutos. Imprime la
revisión descargada para registrarla.


In [ ]:
sys.path.insert(0, '/content/proyecto/scripts')
import runner_d3, ver_traza
modelo = runner_d3.cargar_modelo('microsoft/Phi-3.5-mini-instruct')
print('modelo cargado · revisión', getattr(modelo.modelo.config, '_commit_hash', '?'))

## 5 · Desarrollo
El sistema completo sobre un conjunto de desarrollo: los 52 casos de plantilla
(`casos_desarrollo.jsonl`) o los 21 redactados fuera de las plantillas
(`casos_desarrollo_redactados.jsonl`). Se elige en `CONJUNTO`. Después muestra todas las
fallas y la traza de las cinco primeras, que es lo que se mira para ajustar los prompts.

Si cambiaste un prompt, el runner se niega a retomar el archivo anterior y pide borrarlo: es a
propósito. Corre la celda de borrado y vuelve a correr esta.


In [ ]:
CONJUNTO = 'casos_desarrollo_redactados.jsonl'   # o 'casos_desarrollo.jsonl'
DESARROLLO = f'/content/proyecto/datos/{CONJUNTO}'
# La corrida anterior que está en Drive, para compararla con esta antes de pisarla.
import json
anterior = RESPALDO / 'resultados' / f'agentes__Phi-3.5-mini-instruct__completo__{Path(CONJUNTO).stem}.raw.jsonl'
previas = {}
if anterior.exists():
    previas = {x['id']: x for x in map(json.loads, open(anterior, encoding='utf-8'))}
salida, traza = runner_d3.ejecutar(modelo, 'completo', DESARROLLO)
if previas:
    ahora = {x['id']: x for x in map(json.loads, open(salida, encoding='utf-8'))}
    p = previas[next(iter(previas))]
    print(f"\nContra la corrida anterior (commit {p.get('commit', '?')}, "
          f"{sum(x['acierto_conjunto'] for x in previas.values())}/{len(previas)}):")
    for i in sorted(ahora, key=int):
        if i in previas and ahora[i]['acierto_conjunto'] != previas[i]['acierto_conjunto']:
            print(f"  caso {i:>3}: {'ganó' if ahora[i]['acierto_conjunto'] else 'perdió'}")
respaldar(salida, traza)

import json
filas = [json.loads(l) for l in open(salida, encoding='utf-8')]
fallas = [x for x in filas if not x['acierto_conjunto']]
print(f'\n{len(fallas)} fallas, una por línea (esperado → extraído):\n')
for x in fallas:
    ext = x['extraccion']
    marcas = ' '.join(k for k, v in x['ramas'].items() if v)
    print(f"{x['id']:>3} | {x['pregunta'][:90]}\n"
          f"    {x['ramo_objetivo']['codigo']} {x['esperado']['decision']:11} → "
          f"{ext['ramo']} {ext['periodo']} · {x['prediccion']['decision']} · {marcas}")
print('\nLas primeras cinco con su traza completa:\n')
for x in fallas[:5]:
    print(f"caso {x['id']} · esperado {x['ramo_objetivo']['nombre']} · {x['esperado']}")
    ver_traza.mostrar(traza, int(x['id']), crudo=True)
    print()

In [ ]:
# Borra la corrida de desarrollo, para volver a medir tras cambiar un prompt.
for p in Path('/content/proyecto').glob('*/agentes__*__casos_desarrollo*'):
    p.unlink(); print('borrado', p.name)

## 6 · Etapa final
**Solo después de subir `PROTOCOLO_D3.md`** y de poner su commit en `COMMIT` (sección 1). Corre
las seis configuraciones sobre los tres conjuntos de prueba, y el baseline y el D2 sobre los
redactados fuera de las plantillas (sobre originales y nuevos ya están medidos). Unas dos horas en la T4. Si la
sesión se cae, vuelve a correr las secciones 1 a 4 y esta: el runner retoma donde iba.


In [ ]:
CONJUNTOS = ['casos.jsonl', 'casos_nuevos.jsonl', 'casos_redactados.jsonl']
CONFIGS = ['completo', 'sin_descomposicion', 'sin_herramienta', 'sin_verificador',
           'sin_ejemplos', 'top1']
datos = Path('/content/proyecto/datos')
assert (datos / 'casos_redactados.jsonl').exists(), 'falta casos_redactados.jsonl'
for conjunto in CONJUNTOS:
    for config in CONFIGS:
        salida, traza = runner_d3.ejecutar(modelo, config, str(datos / conjunto))
        respaldar(salida, traza)

In [ ]:
# Baseline few-shot y sistema del D2 sobre los redactados. Cargan su propio modelo.
redactados = str(datos / 'casos_redactados.jsonl')
for args in (['runner.py', '--modelo', 'microsoft/Phi-3.5-mini-instruct',
              '--condicion', 'few_shot', '--casos', redactados],
             ['runner_d2.py', '--variante', 'codigo', '--modo', 'encadenado',
              '--casos', redactados]):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    print('\n'.join(r.stdout.strip().splitlines()[-6:]), r.stderr[-1500:])
for p in Path('/content/proyecto/resultados').glob('*casos_redactados.raw.jsonl'):
    if not p.name.startswith('agentes__'):
        respaldar(p)

## 7 · Bajar los resultados
Un zip con `resultados/` y `trazas/`. También están en Drive, en `genai_d3`.


In [ ]:
from google.colab import files
shutil.make_archive('/content/resultados_d3', 'zip', str(RESPALDO))
files.download('/content/resultados_d3.zip')